# 笔记本 07 — 集成策略与情绪

**阶段 3 · 策略集成 (1 / 2)**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 理解多子策略的**制度加权混合** |
| 2 | 构建子策略权重映射（动量、均值回归、板块轮动） |
| 3 | 使用 `_REGIME_WEIGHTS` 混合因子组合策略 |
| 4 | 应用**情绪叠加**（恐惧与贪婪指数）作为后置乘数 |
| 5 | 实现基于 BTC 占比的**板块轮动** |
| 6 | 将手写集成与生产环境的 `ensemble_combine()` 对比 |

### 前置要求
- NB04（动量）、NB05（均值回归与配对）、NB06（制度检测）

In [ ]:
# ── 初始化 ─────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True})
print("✅ 导入完毕  |  项目根目录:", ROOT)

---
## 1 · 集成策略的概念

没有单一策略能在所有条件下赢利。我们的机器人同时运行**四个**子策略，并根据**当前制度**（来自 NB06）以不同权重组合其输出：

```
┌──────────┐  ┌──────────────┐  ┌─────────────────┐  ┌────────────────┐
│  动量    │  │   均值回归   │  │   板块轮动      │  │    情绪        │
│  权重    │  │    权重      │  │     权重        │  │  （乘数）      │
└────┬─────┘  └──────┬───────┘  └────────┬────────┘  └───────┬────────┘
     │               │                   │                   │
     │  ×混合因子    │  ×混合因子        │  ×混合因子        │
     └───────────────┼───────────────────┘                   │
                     ▼                                       │
              ┌──────────────┐                               │
              │  求和权重    │ ◄──── ×情绪乘数 ──────────────┘
              └──────┬───────┘
                     ▼
              target_weights{symbol: weight}
```

### 制度权重（来自 `bot/strategy/ensemble.py`）

| 制度 | 动量 | 均值回归 | 板块轮动 | 情绪 | 现金 |
|------|------|----------|----------|------|------|
| **牛市** | 0.50 | 0.10 | 0.20 | 0.20 | 0% |
| **震荡** | 0.20 | 0.50 | — | 0.30 | 0% |
| **熊市** | — | 0.30 | — | 0.20 | **50%** |

In [ ]:
# 制度权重表 — 与生产代码完全一致
_REGIME_WEIGHTS = {
    "bull": {
        "momentum": 0.50,
        "sector_rotation": 0.20,
        "sentiment": 0.20,
        "mean_reversion": 0.10,
    },
    "ranging": {
        "mean_reversion": 0.50,
        "sentiment": 0.30,
        "momentum": 0.20,
    },
    "bear": {
        "mean_reversion": 0.30,
        "sentiment": 0.20,
        # 剩余 50% 作为现金持有
    },
}

# 可视化
all_strategies = ["momentum", "mean_reversion", "sector_rotation", "sentiment", "cash"]
colours = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#95a5a6"]
labels_zh = ["动量", "均值回归", "板块轮动", "情绪", "现金"]

data = []
for regime in ["bull", "ranging", "bear"]:
    w = _REGIME_WEIGHTS[regime]
    allocated = sum(w.values())
    row = [w.get(s, 0.0) for s in all_strategies[:-1]] + [max(0, 1.0 - allocated)]
    data.append(row)

weight_df = pd.DataFrame(data, index=["牛市", "震荡", "熊市"], columns=labels_zh)
weight_df.plot.bar(figsize=(10, 5), color=colours, width=0.7, rot=0)
plt.title("各制度下的子策略权重分配")
plt.ylabel("权重")
plt.xlabel("制度")
plt.legend(title="策略", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

weight_df

---
## 2 · 子策略权重映射

每个子策略产出一个**权重映射** — 即 `dict[str, float]`，将币种映射到原始权重。让我们为 6 个币种模拟输出。

In [ ]:
UNIVERSE = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", "AVAXUSDT"]

# 动量信号：按动量得分排名前 3 获得权重
momentum_weights = {
    "SOLUSDT": 0.40,   # 最强动量
    "ETHUSDT": 0.35,
    "AVAXUSDT": 0.25,
}

# 均值回归信号：超卖资产获得权重
mean_reversion_weights = {
    "BTCUSDT": 0.50,   # 最超卖
    "XRPUSDT": 0.30,
    "BNBUSDT": 0.20,
}

# 板块轮动：BTC 占比上升 → 重仓 BTC/ETH
sector_rotation_weights = {
    "BTCUSDT": 0.40,
    "ETHUSDT": 0.25,
    "SOLUSDT": 0.125,
    "BNBUSDT": 0.125,
    "XRPUSDT": 0.05,
    "AVAXUSDT": 0.05,
}

print("动量:       ", momentum_weights)
print("均值回归:   ", mean_reversion_weights)
print("板块轮动:   ", sector_rotation_weights)

---
## 3 · 逐步集成混合

对于给定制度，每个币种的集成计算为：

$$
w_i^{\text{combined}} = \sum_{s \in \text{strategies}} \underbrace{w_i^{(s)}}_{\text{原始权重}} \times \underbrace{\alpha_s^{(\text{regime})}}_{\text{混合因子}}
$$

然后应用情绪乘数 $m$：

$$
w_i^{\text{final}} = w_i^{\text{combined}} \times \text{clamp}(m, 0.5, 1.5)
$$

In [ ]:
def ensemble_combine_manual(
    regime: str,
    momentum_w: dict[str, float],
    mrv_w: dict[str, float],
    sector_w: dict[str, float],
    sentiment_multiplier: float = 1.0,
) -> dict:
    """手写集成——逐步理解每个环节。"""
    regime_key = regime.lower() if regime.lower() in _REGIME_WEIGHTS else "ranging"
    strat_weights = _REGIME_WEIGHTS[regime_key]
    
    # 步骤 1: 用混合因子缩放每个信号映射
    signal_maps = {
        "momentum": momentum_w,
        "mean_reversion": mrv_w,
        "sector_rotation": sector_w,
    }
    
    combined: dict[str, float] = {}
    contributions: dict[str, dict[str, float]] = {}
    
    for sig_name, w_map in signal_maps.items():
        blend = strat_weights.get(sig_name, 0.0)
        if blend <= 0:
            continue
        sig_contrib = {}
        for sym, raw_w in w_map.items():
            c = raw_w * blend
            sig_contrib[sym] = c
            combined[sym] = combined.get(sym, 0.0) + c
        contributions[sig_name] = sig_contrib
    
    # 步骤 2: 情绪叠加
    sent_clamped = max(0.5, min(1.5, sentiment_multiplier))
    if sent_clamped != 1.0:
        combined = {s: w * sent_clamped for s, w in combined.items()}
    
    # 现金分配
    allocated = sum(strat_weights.values())
    cash = max(0.0, 1.0 - allocated)
    
    return {
        "regime": regime_key,
        "target_weights": combined,
        "contributions": contributions,
        "cash_allocation": cash,
    }

# 对三种制度运行
REGIME_NAMES = {"bull": "牛市", "ranging": "震荡", "bear": "熊市"}
for regime in ["bull", "ranging", "bear"]:
    result = ensemble_combine_manual(
        regime, momentum_weights, mean_reversion_weights, sector_rotation_weights
    )
    print(f"\n{'='*50}")
    print(f"制度: {REGIME_NAMES[regime]}  |  现金: {result['cash_allocation']:.0%}")
    print(f"{'='*50}")
    tw = result['target_weights']
    for sym in sorted(tw, key=tw.get, reverse=True):
        print(f"  {sym:12s}: {tw[sym]:.4f}")
    print(f"  {'合计':12s}: {sum(tw.values()):.4f}")

---
## 4 · 可视化贡献分解

堆叠柱状图展示每个策略对各币种的贡献。

In [ ]:
regime = "bull"  # 可更改以查看其他制度
result = ensemble_combine_manual(
    regime, momentum_weights, mean_reversion_weights, sector_rotation_weights
)

contribs = result["contributions"]
all_symbols = sorted(result["target_weights"].keys())

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(all_symbols))
strat_colours = {"momentum": "#3498db", "mean_reversion": "#e74c3c", "sector_rotation": "#2ecc71"}
strat_labels = {"momentum": "动量", "mean_reversion": "均值回归", "sector_rotation": "板块轮动"}

for strat_name, colour in strat_colours.items():
    if strat_name not in contribs:
        continue
    vals = [contribs[strat_name].get(s, 0.0) for s in all_symbols]
    ax.bar(all_symbols, vals, bottom=bottom, label=strat_labels[strat_name], color=colour, alpha=0.8)
    bottom += vals

ax.set_title(f"权重贡献分解 — {REGIME_NAMES[regime]}制度")
ax.set_ylabel("组合权重")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5 · 情绪叠加：恐惧与贪婪指数

[Alternative.me 恐惧与贪婪指数](https://alternative.me/crypto/fear-and-greed-index/) 提供 0–100 的日度情绪读数：

| F&G 分数 | 标签 | 乘数 |
|----------|------|------|
| 0–25 | 极度恐惧 | 1.30（逆向买入） |
| 25–45 | 恐惧 | 1.15 |
| 45–55 | 中性 | 1.00 |
| 55–75 | 贪婪 | 0.85 |
| 75–100 | 极度贪婪 | 0.70（减少敞口） |

乘数按比例缩放**所有**组合权重。

In [ ]:
def fg_to_multiplier(fg_score: float) -> float:
    """将恐惧与贪婪分数 (0-100) 转换为情绪乘数。"""
    if fg_score <= 25:
        return 1.30   # 极度恐惧 → 逆向看涨
    elif fg_score <= 45:
        return 1.15
    elif fg_score <= 55:
        return 1.00
    elif fg_score <= 75:
        return 0.85
    else:
        return 0.70   # 极度贪婪 → 减少敞口

# 遍历 F&G 值
fg_values = range(0, 101, 5)
multipliers = [fg_to_multiplier(fg) for fg in fg_values]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fg_values, multipliers, marker="o", color="#e67e22", lw=2)
ax.axhline(1.0, color="gray", ls="--", lw=0.8)
ax.fill_between(fg_values, multipliers, 1.0,
                where=[m > 1.0 for m in multipliers], alpha=0.15, color="green", label="增加")
ax.fill_between(fg_values, multipliers, 1.0,
                where=[m < 1.0 for m in multipliers], alpha=0.15, color="red", label="减少")
ax.set_xlabel("恐惧与贪婪分数")
ax.set_ylabel("情绪乘数")
ax.set_title("恐惧与贪婪 → 情绪乘数映射")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 情绪对牛市制度下投资组合的影响
scenarios = [
    ("极度恐惧 (F&G=15)", 1.30),
    ("中性 (F&G=50)", 1.00),
    ("极度贪婪 (F&G=85)", 0.70),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (label, mult) in zip(axes, scenarios):
    r = ensemble_combine_manual(
        "bull", momentum_weights, mean_reversion_weights,
        sector_rotation_weights, sentiment_multiplier=mult,
    )
    tw = r["target_weights"]
    syms = sorted(tw.keys())
    vals = [tw[s] for s in syms]
    bars = ax.bar(syms, vals, color="#3498db", alpha=0.8)
    ax.set_title(f"{label}\n(乘数={mult:.2f})")
    ax.set_ylim(0, 0.35)
    ax.tick_params(axis="x", rotation=45)
    total = sum(vals)
    ax.axhline(total / len(syms), ls="--", color="red", lw=0.8)

axes[0].set_ylabel("权重")
fig.suptitle("情绪对牛市制度投资组合的影响", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6 · 基于 BTC 占比的板块轮动

BTC 占比（比特币占加密货币总市值的百分比）反映资金流向：

| 占比趋势 | 轮动制度 | 策略 |
|----------|----------|------|
| 上升（Δ ≥ 0.5%） | `bitcoin_led` | 重仓 BTC/ETH |
| 下降（Δ ≤ -0.5%） | `altcoin_rotation` | 重仓山寨币 |
| 持平 | `neutral` | 均衡配置 |

In [ ]:
# 币种分类（来自生产代码）
_BTC_SYMBOLS = frozenset({"BTCUSDT", "BTCUSD"})
_ETH_SYMBOLS = frozenset({"ETHUSDT", "ETHUSD"})
_LARGE_ALT_PREFIXES = ("SOL", "BNB", "XRP", "ADA", "AVAX", "DOT", "MATIC", "LINK", "DOGE")

def classify_symbol(symbol: str) -> str:
    upper = symbol.upper()
    if upper in _BTC_SYMBOLS: return "btc"
    if upper in _ETH_SYMBOLS: return "eth"
    for prefix in _LARGE_ALT_PREFIXES:
        if upper.startswith(prefix): return "large_alt"
    return "small_alt"

# 板块配置表
SECTOR_ALLOCATIONS = {
    "bitcoin_led":      {"btc": 0.40, "eth": 0.25, "large_alt": 0.25, "small_alt": 0.10},
    "altcoin_rotation": {"btc": 0.15, "eth": 0.20, "large_alt": 0.40, "small_alt": 0.25},
    "neutral":          {"btc": 0.30, "eth": 0.25, "large_alt": 0.30, "small_alt": 0.15},
}

# 可视化配置
alloc_df = pd.DataFrame(SECTOR_ALLOCATIONS).T
alloc_df.index = ["BTC主导", "山寨币轮动", "中性"]
alloc_df.columns = ["BTC", "ETH", "大型山寨币", "小型山寨币"]
alloc_df.plot.bar(figsize=(10, 5), width=0.7,
                  color=["#f1c40f", "#8e44ad", "#3498db", "#e74c3c"], rot=0)
plt.title("按 BTC 占比制度的板块配置")
plt.ylabel("配置权重")
plt.legend(title="板块")
plt.tight_layout()
plt.show()

In [ ]:
def sector_rotation_weights_manual(
    universe: list[str],
    btc_dominance: float,
    prev_dominance: float,
    min_change: float = 0.5,
) -> dict[str, float]:
    """根据 BTC 占比变化计算每个资产的权重。"""
    delta = btc_dominance - prev_dominance
    if delta >= min_change:
        regime = "bitcoin_led"
    elif delta <= -min_change:
        regime = "altcoin_rotation"
    else:
        regime = "neutral"
    
    allocs = SECTOR_ALLOCATIONS[regime]
    
    # 将币种分组到桶中
    buckets: dict[str, list[str]] = {"btc": [], "eth": [], "large_alt": [], "small_alt": []}
    for sym in universe:
        buckets[classify_symbol(sym)].append(sym)
    
    # 桶内等权
    weights = {}
    for bucket, syms in buckets.items():
        if not syms:
            continue
        per_asset = allocs[bucket] / len(syms)
        for sym in syms:
            weights[sym] = per_asset
    
    return weights

# 示例：BTC 占比上升
universe = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT", "AVAXUSDT", "DOGEUSDT"]
for dom, prev, label in [(55.0, 53.0, "BTC主导 (+2%)"), (50.0, 50.3, "中性"), (48.0, 50.0, "山寨币轮动 (-2%)")]:
    w = sector_rotation_weights_manual(universe, dom, prev)
    print(f"\n{label}:")
    for sym in sorted(w, key=w.get, reverse=True):
        print(f"  {sym:12s}: {w[sym]:.3f}")

---
## 7 · 跨制度的完整集成示例

追踪完整管线：制度 → 混合 → 情绪 → 最终权重。

In [ ]:
# 模拟 30 天，制度和情绪变化
np.random.seed(42)
days = 30
regimes = ["bull"] * 12 + ["ranging"] * 10 + ["bear"] * 8
fg_scores = np.clip(50 + np.cumsum(np.random.randn(days) * 5), 5, 95)

# 跟踪每日股权配置（非现金权重之和）
daily_allocation = []
daily_cash = []
daily_top_asset = []

for i in range(days):
    mult = fg_to_multiplier(fg_scores[i])
    r = ensemble_combine_manual(
        regimes[i], momentum_weights, mean_reversion_weights,
        sector_rotation_weights, sentiment_multiplier=mult,
    )
    tw = r["target_weights"]
    daily_allocation.append(sum(tw.values()))
    daily_cash.append(r["cash_allocation"])
    daily_top_asset.append(max(tw, key=tw.get) if tw else "N/A")

regime_colours = {"bull": "#2ecc71", "ranging": "#f39c12", "bear": "#e74c3c"}
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# 制度
for i in range(days):
    ax1.axvspan(i - 0.5, i + 0.5, alpha=0.3, color=regime_colours[regimes[i]])
ax1.set_ylabel("制度")
ax1.set_yticks([])
patches = [mpatches.Patch(color=c, alpha=0.3, label=REGIME_NAMES[r]) for r, c in regime_colours.items()]
ax1.legend(handles=patches, loc="upper right")
ax1.set_title("30 天集成策略模拟")

# F&G 分数
ax2.plot(range(days), fg_scores, color="#e67e22", lw=2)
ax2.axhline(50, color="gray", ls="--", lw=0.8)
ax2.fill_between(range(days), fg_scores, 50,
                 where=fg_scores < 50, alpha=0.15, color="green")
ax2.fill_between(range(days), fg_scores, 50,
                 where=fg_scores > 50, alpha=0.15, color="red")
ax2.set_ylabel("F&G 分数")
ax2.set_ylim(0, 100)

# 股权配置
ax3.bar(range(days), daily_allocation, color="#3498db", alpha=0.7, label="股权")
ax3.bar(range(days), daily_cash, bottom=daily_allocation, color="#95a5a6", alpha=0.5, label="现金")
ax3.set_ylabel("总配置")
ax3.set_xlabel("天")
ax3.legend()

plt.tight_layout()
plt.show()

---
## 8 · 生产代码：`ensemble_combine()`

In [ ]:
from bot.strategy.ensemble import ensemble_combine, _REGIME_WEIGHTS as prod_weights

# 与手写实现对比
for regime in ["bull", "ranging", "bear"]:
    prod_result = ensemble_combine(
        regime,
        momentum_weights=momentum_weights,
        mean_reversion_weights=mean_reversion_weights,
        sector_rotation_weights=sector_rotation_weights,
        sentiment_multiplier=1.0,
    )
    manual_result = ensemble_combine_manual(
        regime, momentum_weights, mean_reversion_weights,
        sector_rotation_weights, sentiment_multiplier=1.0,
    )
    
    match = all(
        abs(prod_result.target_weights.get(s, 0) - manual_result["target_weights"].get(s, 0)) < 1e-9
        for s in set(list(prod_result.target_weights) + list(manual_result["target_weights"]))
    )
    print(f"{REGIME_NAMES[regime]:8s} — 一致: {'✅' if match else '❌'}  |  现金: {prod_result.cash_allocation:.0%}")

In [ ]:
# 生产板块轮动
from bot.signals.sector_rotation import sector_rotation_weights as prod_sector_weights

prod_sw = prod_sector_weights(universe, btc_dominance=55.0, previous_dominance=53.0)
manual_sw = sector_rotation_weights_manual(universe, 55.0, 53.0)

print("生产 vs 手写 板块权重:")
for sym in sorted(prod_sw.keys()):
    p = prod_sw.get(sym, 0)
    m = manual_sw.get(sym, 0)
    status = "✅" if abs(p - m) < 1e-9 else "❌"
    print(f"  {sym:12s}: 生产={p:.4f}  手写={m:.4f}  {status}")

---
## 9 · 敏感性：制度权重如何影响收益

如果我们改变制度权重分配，结果会如何？让我们来模拟。

In [ ]:
# 为每种策略模拟日收益
np.random.seed(123)
n_days = 200

# 牛市：动量表现好；熊市：均值回归存活
strategy_returns = {
    "momentum":       np.random.normal(0.003, 0.02, n_days),
    "mean_reversion": np.random.normal(0.001, 0.01, n_days),
    "sector_rotation":np.random.normal(0.002, 0.015, n_days),
}

# 熊市中动量表现差（最后 50 天）
strategy_returns["momentum"][-50:] = np.random.normal(-0.005, 0.03, 50)
strategy_returns["mean_reversion"][-50:] = np.random.normal(0.002, 0.01, 50)

# 制度序列
sim_regimes = ["bull"] * 80 + ["ranging"] * 70 + ["bear"] * 50

# 1. 静态等权（不适应制度）
static_returns = []
for i in range(n_days):
    r = sum(strategy_returns[s][i] * (1/3) for s in strategy_returns)
    static_returns.append(r)

# 2. 制度自适应权重
adaptive_returns = []
for i in range(n_days):
    rw = _REGIME_WEIGHTS[sim_regimes[i]]
    r = sum(strategy_returns[s][i] * rw.get(s, 0.0) for s in strategy_returns)
    adaptive_returns.append(r)

static_equity  = np.exp(np.cumsum(static_returns))
adaptive_equity = np.exp(np.cumsum(adaptive_returns))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(static_equity, label="静态等权", lw=1.5, alpha=0.8)
ax.plot(adaptive_equity, label="制度自适应", lw=1.5, alpha=0.8)

# 着色制度阶段
for i in range(n_days):
    ax.axvspan(i - 0.5, i + 0.5, alpha=0.05, color=regime_colours[sim_regimes[i]])

ax.set_title("静态 vs 制度自适应集成")
ax.set_xlabel("天")
ax.set_ylabel("净值（对数尺度）")
ax.legend()
plt.tight_layout()
plt.show()

print(f"静态最终净值:   {static_equity[-1]:.4f}")
print(f"自适应最终净值: {adaptive_equity[-1]:.4f}")
print(f"优势:           {(adaptive_equity[-1]/static_equity[-1] - 1)*100:.1f}%")

---
## 10 · 关键要点

| 概念 | 详情 |
|------|------|
| **集成混合** | 用制度依赖的混合因子缩放每个子策略权重后求和 |
| **制度权重** | 牛市偏好动量(0.50)；震荡偏好均值回归(0.50)；熊市持有 50% 现金 |
| **情绪叠加** | 恐惧与贪婪指数 → 乘数 [0.5, 1.5] 在混合后应用 |
| **板块轮动** | BTC 占比变化 → 重仓 BTC/ETH vs 重仓山寨币 |
| **现金配置** | 熊市制度明确扣留 50% 现金 — 任何信号都无法覆盖 |

### 集成管线

```
detect_regime() → 制度 (牛市 / 震荡 / 熊市)
    │
    ├── _REGIME_WEIGHTS[regime] → 混合因子
    │
    ├── 动量权重 × blend_factor["momentum"]
    ├── 均值回归权重 × blend_factor["mean_reversion"]
    ├── 板块权重 × blend_factor["sector_rotation"]
    │
    ├── 求和 → 组合权重
    │
    ├── × 情绪乘数(F&G)
    │
    └── target_weights + cash_allocation
```

---
## 🔬 练习

1. **自定义制度权重：** 设计你自己的 `_REGIME_WEIGHTS`，增加第 4 种制度"volatile_bull"（动量=0.30, 均值回归=0.20, 板块=0.20, 情绪=0.30）。测试效果。

2. **动态情绪：** 使用线性映射替代离散分桶：`multiplier = 1.5 - (fg_score / 100)`。这如何改变投资组合行为？

3. **加权平均 vs 最大投票：** 用最大投票方法替代求和——每个币种只保留其最高贡献者。比较结果。

4. **回测集成：** 使用 NB09 的回测引擎，在 1 年 BTC 数据上比较制度自适应 vs 静态权重集成。

---
## ✅ 知识检查

1. 为什么熊市制度持有 50% 现金而不是将其分配给均值回归？
2. 情绪乘数如何与现金配置交互？（提示：不交互。）
3. 如果 momentum_weights 有 3 个资产，sector_rotation_weights 有 6 个，组合输出中有多少个资产？
4. 如果你传入一个未知制度（如"crash"）给 `ensemble_combine()`，会发生什么？
5. 为什么情绪乘数被限制在 [0.5, 1.5]？

---
## 🔗 下一步

**[NB08 — 投资组合优化与风控 →](08_投资组合优化与风控.ipynb)**

我们将把集成的 `target_weights` 通过权重归一化、仓位限制、现金底线和熔断机制进行处理。